In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = (SparkSession.builder
            .appName("Preprocess")
            .getOrCreate()
 )

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-26d6719b-a169-493f-8834-106317309733;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 122ms :: artifacts dl 4ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------

In [3]:
df = spark.read.parquet('/user/data/raw/*.parquet')

n = len(df.inputFiles())
print(f"Number of raw parquet files: {n}")

print(f"Total number of rows: {df.count()}")

df.printSchema()

Number of raw parquet files: 6


Total number of rows: 24083384
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [4]:
df = df.dropna(subset=[
  "PULocationID", "DOLocationID", 
  "tpep_pickup_datetime", "tpep_dropoff_datetime"
])

df = df.filter(
  (F.col("trip_distance") > 0) &
  (F.col("passenger_count") > 0) &
  (F.col("fare_amount") > 0) &
  (F.col("total_amount") > 0)
)

df = df.withColumn(
  "trip_duration",
  (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
)

df = df.filter(
  (F.col("trip_duration") > 1) &
  (F.col("trip_duration") < 1440)
)

df = df.withColumn("hour", F.hour("tpep_pickup_datetime")) \
        .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")) \
        .withColumn("month", F.month("tpep_pickup_datetime")) \
        .withColumn("is_weekend", F.when(F.col("day_of_week").isin([1, 7]), 1).cast("int"))

df = df.withColumn(
  "average_speed",
  F.col("trip_distance") / (F.col("trip_duration") / 60)
)

df = df.filter(
  (F.col("average_speed") > 1) &
  (F.col("average_speed") < 100)
)

def remove_outliers(df, column):
  quantiles = df.approxQuantile(column, [0.25, 0.75], 0.05)
  Q1, Q3 = quantiles
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  return df.filter((F.col(column) >= lower_bound) & (F.col(column) <= upper_bound))

for col in ["trip_distance", "fare_amount", "total_amount", "trip_duration"]:
  df = remove_outliers(df, col)

df = df.withColumn(
  "tip_ratio",
  F.when(F.col("fare_amount") > 0, 
          F.col("tip_amount") / F.col("fare_amount"))
  .otherwise(0)
)

df = df.withColumn(
  "cost_per_minute",
  F.col("total_amount") / F.col("trip_duration")
)

df = df.withColumn(
  "is_credit",
  (F.col("payment_type") == 1).cast("int")
)

df = df.withColumn(
  "is_airport_trip",
  (F.col("Airport_fee") > 0).cast("int")
)

zone_demand = df.groupBy("PULocationID").count() \
                .withColumnRenamed("count", "zone_demand")
df = df.join(zone_demand, on="PULocationID", how="left")

df = df.select(
  "PULocationID", "DOLocationID",
  "trip_distance", "trip_duration",
  "average_speed",
  "cost_per_minute", "tip_ratio",
  "hour", "day_of_week", "month", "is_weekend",
  "zone_demand"
)

print(df.count())
df.printSchema()

14244078
root
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- trip_duration: double (nullable = true)
 |-- average_speed: double (nullable = true)
 |-- cost_per_minute: double (nullable = true)
 |-- tip_ratio: double (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- zone_demand: long (nullable = true)



In [5]:
df.repartition("month") \
  .write \
  .mode("overwrite") \
  .partitionBy("month") \
  .parquet('/user/data/preprocess/kmeans/')

In [6]:
df.select("month").distinct().show()

+-----+
|month|
+-----+
|    1|
|    3|
|    2|
|    4|
|    5|
|    6|
|   12|
+-----+

